In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# TEST phenotypes

In [2]:
dpheno = pd.read_csv("../44_train_and_test/00_combine_samples.tsv", sep="\t", header=0)
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

(11433, 9)


,genotype,POP_ID,SITE_ID,call_rate,good_genotype,value,Trait_name,SET,POP_SITE,sample
0,E60A-3_1_84_089,1530,AC,NaN,False,513.677825,Average_Ring_Density,TEST,1530_AC,1530_AC
1,E60A-3_3_84_091,1530,AC,NaN,False,554.737825,Average_Ring_Density,TEST,1530_AC,1530_AC
2,E60A-3_5_84_090,1530,AC,NaN,False,580.927825,Average_Ring_Density,TEST,1530_AC,1530_AC
3,E60A-3_6_84_093,1530,AC,NaN,False,575.127825,Average_Ring_Density,TEST,1530_AC,1530_AC
4,E60A-3_7_84_094,1530,AC,NaN,False,567.897825,Average_Ring_Density,TEST,1530_AC,1530_AC


In [3]:
#dpheno[dpheno["n"] < 3].shape

In [4]:
#dpheno = dpheno[dpheno['n']>2].reset_index(drop=True)
#print(dpheno.shape)

# TRAIN phenotypes

In [5]:
tpheno = pd.read_csv("../44_train_and_test/00_combine_samples.tsv", sep="\t", header=0)
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
print(tpheno.shape)
tpheno["sample"] = tpheno["POP_SITE"]

#print(tpheno[tpheno["n"] < 3].shape)

#tpheno = tpheno[tpheno['n']>2].reset_index(drop=True)
print(tpheno.shape)

(18282, 9)
(18282, 10)


# Combining genetic offset sets

In [6]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]


dsets = ["1000"]
#dsets = ["100","1000","10000", "lfmm","RDA","RDAcorrected"]
#dsets = ["100","1000","lfmm"]
dsets

['1000']

In [7]:
D = []

for trait in traits:
    for dset in dsets:
        for garden in gardens:
            file_path = "../991_run_rda_separate/results/01_run_gradient_forest_" + garden + "_" + dset + "_" + trait + ".tsv"
            if os.path.exists(file_path):
                df_ = pd.read_csv(file_path, sep="\t", header=0)
                df_["marker"] = dset
                df_["trait"] = trait
                df_["garden"] = garden
                D.append(df_)
dD = pd.concat(D)
print(dD.shape)
dD.head()

(2019, 17)


,Trait_name,sample,SITE_ID,group,PC1,PC2,PC3,now_at1,now_at2,now_at3,new_at1,new_at2,new_at3,offset_rda,marker,trait,garden
0,Height,2209_PR,PR,West,12.351204,-1.598833,0.851360,-0.143915,-0.029449,0.039853,-0.104196,-0.006909,-0.020072,0.075343,1000,Height,PR
1,Height,325_PR,PR,Central,2.506557,-3.166880,1.958590,0.203114,-0.030488,0.010425,-0.104196,-0.006909,-0.020072,0.309719,1000,Height,PR
2,Height,4351_PR,PR,Central,-6.244846,1.531312,-1.399408,0.062366,0.008586,-0.042347,-0.104196,-0.006909,-0.020072,0.168758,1000,Height,PR
3,Height,4420_PR,PR,West,6.170760,3.960606,-2.183338,-0.322852,0.015656,0.002525,-0.104196,-0.006909,-0.020072,0.220976,1000,Height,PR
4,Height,6805_PR,PR,East,0.111999,-2.575310,0.678848,0.137265,-0.026911,-0.009894,-0.104196,-0.006909,-0.020072,0.242502,1000,Height,PR


In [8]:
dD_sub = dD[["sample","SITE_ID","offset_rda","marker","trait"]].reset_index(drop=True)
dD_sub.head()

,sample,SITE_ID,offset_rda,marker,trait
0,2209_PR,PR,0.075343,1000,Height
1,325_PR,PR,0.309719,1000,Height
2,4351_PR,PR,0.168758,1000,Height
3,4420_PR,PR,0.220976,1000,Height
4,6805_PR,PR,0.242502,1000,Height


### Test

In [9]:
dD_merge = pd.merge(dD_sub, dpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

(11100, 13)


,sample,SITE_ID,offset_rda,marker,trait,genotype,POP_ID,call_rate,good_genotype,value,Trait_name,SET,POP_SITE
0,2209_PR,PR,0.075343,1000,Height,443-G348-1-17-4,2209,NaN,False,975.525258,Height,TEST,2209_PR
1,2209_PR,PR,0.075343,1000,Height,741-G348-2-17-1,2209,NaN,False,1175.525258,Height,TEST,2209_PR
2,2209_PR,PR,0.075343,1000,Height,742-G348-2-17-2,2209,NaN,False,1215.525258,Height,TEST,2209_PR
3,2209_PR,PR,0.075343,1000,Height,773-G348-3-17-4,2209,NaN,False,555.525258,Height,TEST,2209_PR
4,325_PR,PR,0.309719,1000,Height,504-G348-1-23-1,325,NaN,False,1292.391145,Height,TEST,325_PR


In [10]:
# Filtering out small pop_sites (n<4) from dataframe
dD_pops = dD_merge.groupby(['POP_SITE','trait']).agg(n = ('genotype','count')).sort_values(by='n', ascending=True).reset_index()
small_group = dD_pops.loc[dD_pops["n"]>=2,['POP_SITE','trait']].reset_index(drop=True)
small_group

dD_merge_sub = pd.merge(small_group, dD_merge, on = ['POP_SITE','trait'], how = 'left')
print(dD_merge_sub.shape)
dD_merge_sub.head()

(11073, 13)


,POP_SITE,trait,sample,SITE_ID,offset_rda,marker,genotype,POP_ID,call_rate,good_genotype,value,Trait_name,SET
0,6930_CH,Biomass_Increment_2015,6930_CH,CH,0.190345,1000,E353-B3_5_11_6930_694,6930,0.907426,True,0.024961,Biomass_Increment_2015,TEST
1,6930_CH,Biomass_Increment_2015,6930_CH,CH,0.190345,1000,E353-B3_5_16_6930_696,6930,0.896530,True,0.182145,Biomass_Increment_2015,TEST
2,332_ML,Biomass_Increment_1980,332_ML,ML,0.104127,1000,E353-B1_6_11_332_508,332,0.865086,True,0.140273,Biomass_Increment_1980,TEST
3,332_ML,Biomass_Increment_1980,332_ML,ML,0.104127,1000,E353-B1_6_14_332_509,332,0.887710,True,0.685433,Biomass_Increment_1980,TEST
4,332_ML,Rr,332_ML,ML,0.094576,1000,E353-B1_4_6_332_273,332,0.879689,True,0.034801,Rr,TEST


### Subsetting TEST samples for calculaing mean trait per POP_SITE

In [19]:
T = []
for n in [2,4,6,8,10,12,14,16,18,20]:
    for i in [1,2,3,4,5]:
        df_ = dD_merge_sub.groupby(['POP_SITE','trait'],group_keys=False)[['POP_SITE', 'trait', 'sample', 'SITE_ID', 'offset_rda', 'marker','genotype', 'POP_ID', 'value','SET']].apply(
            lambda x: x.sample(n=min(len(x), n),random_state=42+i))
        df_means = df_.groupby(['POP_SITE', 'trait', 'sample', 'SITE_ID', 'offset_rda', 'marker','POP_ID','SET']).agg(
            mean = ('value','mean'), sd = ('value','std'),
            n_test = ('value','count')).reset_index()
        df_means['n'] = n
        df_means['i'] = i
        T.append(df_means)
dT = pd.concat(T, ignore_index=True)
dT.head()

,POP_SITE,trait,sample,SITE_ID,offset_rda,marker,POP_ID,SET,mean,sd,n_test,n,i
0,1329_CH,Average_Ring_Density,1329_CH,CH,0.067857,1000,1329,TEST,491.301667,50.626489,2,2,1
1,1329_CH,Biomass_Increment,1329_CH,CH,0.076472,1000,1329,TEST,3.956788,0.013860,2,2,1
2,1329_CH,Biomass_Increment_1985,1329_CH,CH,0.115901,1000,1329,TEST,0.982286,0.530467,2,2,1
3,1329_CH,Biomass_Increment_1990,1329_CH,CH,0.063636,1000,1329,TEST,2.154921,0.064434,2,2,1
4,1329_CH,Biomass_Increment_1995,1329_CH,CH,0.076030,1000,1329,TEST,2.479106,0.272032,2,2,1


### Calculating rho

In [20]:
def spearman_group(df, col_a, col_b, group_cols):
    results = []
    for name, g in df.groupby(group_cols):
        if len(g) < 3:
            rho, p = np.nan, np.nan
        elif ((len(g[col_a].dropna())<3) or (len(g[col_b].dropna())<3)):
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(g[col_a], g[col_b], nan_policy='omit')
        # name is tuple if multiple group columns
        name = name if isinstance(name, tuple) else (name,)
        results.append((*name, rho, p, len(g)))
            
    res_df = pd.DataFrame(results, columns=list(group_cols) + ['spearman_rho', 'p_value', 'n_samples'])
    
    # Adjust p-values
    mask = res_df['p_value'].notna()
    res_df['p_adj'] = np.nan
    if mask.any():
        res_df.loc[mask, 'p_adj'] = multipletests(res_df.loc[mask, 'p_value'], method='fdr_bh')[1]

    return res_df

### Test

In [21]:
dT.head()

,POP_SITE,trait,sample,SITE_ID,offset_rda,marker,POP_ID,SET,mean,sd,n_test,n,i
0,1329_CH,Average_Ring_Density,1329_CH,CH,0.067857,1000,1329,TEST,491.301667,50.626489,2,2,1
1,1329_CH,Biomass_Increment,1329_CH,CH,0.076472,1000,1329,TEST,3.956788,0.013860,2,2,1
2,1329_CH,Biomass_Increment_1985,1329_CH,CH,0.115901,1000,1329,TEST,0.982286,0.530467,2,2,1
3,1329_CH,Biomass_Increment_1990,1329_CH,CH,0.063636,1000,1329,TEST,2.154921,0.064434,2,2,1
4,1329_CH,Biomass_Increment_1995,1329_CH,CH,0.076030,1000,1329,TEST,2.479106,0.272032,2,2,1


In [22]:
df_cors = spearman_group(dT, 'offset_rda', 'mean', ['SITE_ID','trait','marker','n','i'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,marker,n,i,spearman_rho,p_value,n_samples,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,2,1,-0.794355,9.572199e-08,31,0.000054,minus,TEST
1,AC,Average_Ring_Density,1000,2,2,-0.597984,3.812530e-04,31,0.004320,minus,TEST
2,AC,Average_Ring_Density,1000,2,3,-0.632258,1.357862e-04,31,0.002472,minus,TEST
3,AC,Average_Ring_Density,1000,2,4,-0.638710,1.102587e-04,31,0.002379,minus,TEST
4,AC,Average_Ring_Density,1000,2,5,-0.657661,5.813716e-05,31,0.001566,minus,TEST


In [23]:
df_cors.to_csv("14_garden_check_test_rda.tsv", sep="\t", header=True, index=False)

# END